In [1]:
# ==========================================
# PHASE 1: IMPORTS WITH COMPREHENSIVE ERROR HANDLING
# ==========================================

import os
import json
import logging
from pathlib import Path
from typing import Optional
HEADERS = {
    "x-rapidapi-key": os.getenv('RAPIDAPI_KEY', ''),
    "x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Load environment variables with error handling
try:
    from dotenv import load_dotenv
    load_dotenv()
    logger.info("✅ Environment variables loaded from .env file")
except ImportError as e:
    logger.warning(f"⚠️ python-dotenv not installed: {e}. Using fallback environment.")
except Exception as e:
    logger.error(f"❌ Failed to load .env file: {e}")

# Initialize environment variables with fallback values
env_vars = {
    'GOOGLE_API_KEY': 'GOOGLE_API_KEY',
    'TAVILY_API_KEY': 'TAVILY_API_KEY',
    'GROQ_API_KEY': 'GROQ_API_KEY',
    'RAPIDAPI_KEY': 'RAPIDAPI_KEY',
    'WEATHER_API_KEY': 'WEATHER_API_KEY'
}

for var_name, var_key in env_vars.items():
    try:
        value = os.getenv(var_key)
        if value:
            os.environ[var_name] = value
            logger.info(f"✅ {var_name} loaded successfully")
        else:
            logger.warning(f"⚠️ {var_name} not found in environment. Using default.")
    except Exception as e:
        logger.error(f"❌ Error loading {var_name}: {e}")

# Initialize LangChain chat model with error handling
try:
    from langchain.chat_models import init_chat_model
    model = init_chat_model(model='openai/gpt-oss-120b', model_provider="groq")
    logger.info("✅ LangChain model initialized successfully (Groq)")
except ImportError as e:
    logger.error(f"❌ Failed to import init_chat_model: {e}")
    raise SystemExit("Required dependency 'langchain' not found. Please install it.")
except Exception as e:
    logger.error(f"❌ Failed to initialize chat model: {e}")
    raise SystemExit(f"Chat model initialization failed: {e}")
# model = init_chat_model(model='gemini-3-pro-preview',model_provider="google_genai",base_url="https://api.ai-gateway.tigeranalytics.com")

# ================

2026-03-28 05:39:52,199 - INFO - ✅ Environment variables loaded from .env file
2026-03-28 05:39:52,199 - INFO - ✅ GOOGLE_API_KEY loaded successfully
2026-03-28 05:39:52,200 - INFO - ✅ TAVILY_API_KEY loaded successfully
2026-03-28 05:39:52,200 - INFO - ✅ GROQ_API_KEY loaded successfully
2026-03-28 05:39:52,201 - INFO - ✅ RAPIDAPI_KEY loaded successfully
2026-03-28 05:39:52,202 - INFO - ✅ WEATHER_API_KEY loaded successfully
2026-03-28 05:40:04,637 - INFO - ✅ LangChain model initialized successfully (Groq)


In [ ]:
# def get_user_selections(file_path) -> list:
#     """
#     Modified to extract Player IDs and Names from Structured Pydantic/JSON logs.
#     Handles the new 'FantasyPlayer' string format and dictionary format.
#     """
#     try:
#         with open(file_path, "r", encoding='utf-8') as f:
#             logs = json.load(f)

#         final_teams = []
#         for entry in logs:
#             raw_data = entry.get("team", "")
#             players = []

#             # CASE A: Data is already a structured dictionary (Best case)
#             if isinstance(raw_data, dict) and "players" in raw_data:
#                 for p in raw_data["players"]:
#                     players.append({
#                         "id": str(p.get("player_id", p.get("id"))),
#                         "name": p.get("name", "Unknown").replace("\u202f", " ").strip()
#                     })

#             # CASE B: Data is a String (Pydantic __repr__ or Markdown)
#             else:
#                 raw_text = str(raw_data)

#                 # 1. Check for Pydantic String format: FantasyPlayer(player_id='10698', name='Dane Cleaver'...)
#                 pydantic_matches = re.findall(r"player_id='(\d+)',\s*name='([^']+)'", raw_text)

#                 if pydantic_matches:
#                     for p_id, p_name in pydantic_matches:
#                         players.append({
#                             "id": p_id,
#                             "name": p_name.replace("\u202f", " ").strip()
#                         })

#                 # 2. Fallback to old Markdown Table format if Pydantic check fails
#                 else:
#                     table_matches = re.findall(r"\|\s*(\d+)\s*\|\s*([^|]+?)\s*\|", raw_text)
#                     for p_id, p_name in table_matches:
#                         clean_name = p_name.replace("**", "").replace("\u202f", " ").strip()
#                         # Extract ID if it's hidden in the name like "Name (123)"
#                         id_search = re.search(r"\((\d+)\)", clean_name)
#                         actual_id = id_search.group(1) if id_search else p_id
#                         final_name = re.sub(r"\s*\(\d+\)", "", clean_name).strip()

#                         players.append({"id": actual_id, "name": final_name})

#             # Add to final list if players were found
#             final_teams.append({
#                 "user": entry.get("user"),
#                 "agent": entry.get("agent_type"),
#                 "players": players
#             })

#         return final_teams

#     except Exception as e:
#         return [{"error": f"Failed to parse logs: {str(e)}"}]

# # --- Execution ---
# play = get_user_selections('/mnt/c/workspaces/agent_project/notebooks1/agent_selections_log.json')
# print(json.dumps(play[:2], indent=2))

In [ ]:
import json
import re
import time
import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from langchain.tools import tool
# from langchain_anthropic import ChatAnthropic

# ── Google Sheets Setup ────────────────────────────────────────────────────────
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_name(
    "/mnt/c/workspaces/agent_project/myagentproject-491107-729a12dc5f1d.json", scope
)
client = gspread.authorize(creds)
spreadsheet    = client.open("Cricket_Project_DB.")
agent_response_sheet = spreadsheet.worksheet("Agent_Response")

# ── Tool 1: Fetch Match Stats from API ────────────────────────────────────────
@tool
def fetch_match_stats(match_id: str) -> dict:
    """
    Fetches raw scorecard data from the CricBuzz API and organizes it
    into batting and bowling performance maps for fantasy calculation.
    """
    url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/scard"
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        data = response.json()
        performance_map = {}

        for innings in data.get('scorecard', []):

            # ── Parse Batting ──────────────────────────────────────────────
            for bat in innings.get('batsman', []):
                p_id = str(bat['id'])
                out_desc = bat.get('outdec', "").lower()
                lbw_bowled = 1 if ("lbw" in out_desc or out_desc.startswith("b ")) else 0

                bat_stats = {
                    "name": bat.get('name'),
                    "batting": {
                        "runs":             int(bat.get('runs', 0)),
                        "balls":            int(bat.get('balls', 0)),
                        "fours":            int(bat.get('fours', 0)),
                        "sixes":            int(bat.get('sixes', 0)),
                        "sr":               float(bat.get('strkrate', 0)),
                        "is_out":           out_desc != "not out",
                        "lbw_bowled_victim": lbw_bowled
                    }
                }

                if p_id not in performance_map:
                    performance_map[p_id] = bat_stats
                else:
                    performance_map[p_id]["batting"] = bat_stats["batting"]

            # ── Parse Bowling ──────────────────────────────────────────────
            # Also count LBW/Bowled victims for THIS bowler from batting list
            lbw_bowled_counts = {}
            for bat in innings.get('batsman', []):
                out_desc = bat.get('outdec', "").lower()
                bowler_name = bat.get('bowler', "").strip().lower()
                if "lbw" in out_desc or out_desc.startswith("b "):
                    lbw_bowled_counts[bowler_name] = lbw_bowled_counts.get(bowler_name, 0) + 1

            for bowl in innings.get('bowler', []):
                p_id = str(bowl['id'])
                bowler_name = bowl.get('name', "").strip().lower()

                bowl_stats = {
                    "bowling": {
                        "wickets":        int(bowl.get('wickets', 0)),
                        "overs":          float(bowl.get('overs', 0)),
                        "maidens":        int(bowl.get('maidens', 0)),
                        "runs_conceded":  int(bowl.get('runs', 0)),
                        "econ":           float(bowl.get('economy', 0)),
                        "dots":           int(bowl.get('dots', 0)),
                        "lbw_bowled_count": lbw_bowled_counts.get(bowler_name, 0)  # ✅ Fixed
                    }
                }

                if p_id not in performance_map:
                    performance_map[p_id] = {"name": bowl.get('name'), "bowling": bowl_stats["bowling"]}
                else:
                    performance_map[p_id]["bowling"] = bowl_stats["bowling"]

        return performance_map

    except Exception as e:
        return {"error": str(e)}


# ── Fantasy Points Calculator (2026 Rules) ────────────────────────────────────
def calculate_fantasy_points_2026(p_data: dict) -> int:
    """
    Calculates T20 Fantasy Points using 2026 standard rules.
    """
    pts = 0.0

    # ── Batting ───────────────────────────────────────────────────────────────
    if "batting" in p_data:
        bat   = p_data["batting"]
        runs  = int(bat.get('runs', 0))
        balls = int(bat.get('balls', 0))
        fours = int(bat.get('fours', 0))
        sixes = int(bat.get('sixes', 0))
        sr    = float(bat.get('sr', 0))

        pts += runs
        pts += (fours * 1)
        pts += (sixes * 2)

        if runs >= 100:  pts += 16
        elif runs >= 50: pts += 8
        elif runs >= 30: pts += 4

        if runs == 0 and balls > 0:
            pts -= 2  # Duck penalty

        if balls >= 10:
            if sr > 170:            pts += 6
            elif 150 < sr <= 170:   pts += 4
            elif 130 <= sr <= 150:  pts += 2
            elif 60 <= sr <= 70:    pts -= 2
            elif 50 <= sr < 60:     pts -= 4
            elif sr < 50:           pts -= 6

    # ── Bowling ───────────────────────────────────────────────────────────────
    if "bowling" in p_data:
        bowl            = p_data["bowling"]
        wickets         = int(bowl.get('wickets', 0))
        overs           = float(bowl.get('overs', 0))
        econ            = float(bowl.get('econ', 0))
        maidens         = int(bowl.get('maidens', 0))
        lbw_bowled_count = int(bowl.get('lbw_bowled_count', 0))

        pts += (wickets * 25)
        pts += (lbw_bowled_count * 8)   # LBW/Bowled bonus per victim

        if wickets >= 5:   pts += 16
        elif wickets >= 4: pts += 8
        elif wickets >= 3: pts += 4

        pts += (maidens * 12)

        if overs >= 2.0:
            if econ < 5:             pts += 6
            elif 5 <= econ < 6:      pts += 4
            elif 6 <= econ <= 7:     pts += 2
            elif 10 <= econ < 11:    pts -= 2
            elif 11 <= econ <= 12:   pts -= 4
            elif econ > 12:          pts -= 6

    return int(pts)


# ── Tool 2: Read User Selections FROM Google Sheet ────────────────────────────
@tool
def get_user_selections_from_sheet(match_id: str) -> list:
    """
    Reads player selections from the Agent_Response Google Sheet.
    Filters rows by Match_ID and parses the response JSON to extract players.
    Returns a list of dicts: [{user, agent, players: [{id, name}]}]
    """
    try:
        all_values = agent_response_sheet.get_all_values()
        headers    = all_values[0]   # ['Player_id', 'agent_type', 'response', 'Match_id']
        data_rows  = all_values[1:]  # all data rows

        teams = []
        for row in data_rows:
            if not any(row):
                continue

            row_player_id = row[0].strip()
            row_agent     = row[1].strip()
            row_response  = row[2].strip()
            row_match_id  = str(row[3]).strip()

            # ── Filter by match_id ─────────────────────────────────────────
            if row_match_id != str(match_id).strip():
                continue

            players = []

            # ── Parse response: try JSON first ─────────────────────────────
            if row_response.startswith('{'):
                try:
                    parsed = json.loads(row_response)
                    for p in parsed.get("players", []):
                        players.append({
                            "id":   str(p.get("player_id", p.get("id", ""))),
                            "name": p.get("name", "Unknown").replace("\u202f", " ").strip()
                        })
                except json.JSONDecodeError:
                    pass

            # ── Parse response: Pydantic repr style ───────────────────────
            if not players:
                pydantic_matches = re.findall(r"player_id='(\d+)',\s*name='([^']+)'", row_response)
                for p_id, p_name in pydantic_matches:
                    players.append({
                        "id":   p_id,
                        "name": p_name.replace("\u202f", " ").strip()
                    })

            # ── Parse response: Markdown table fallback ────────────────────
            if not players:
                table_matches = re.findall(r"\|\s*(\d+)\s*\|\s*([^|]+?)\s*\|", row_response)
                for p_id, p_name in table_matches:
                    clean_name = p_name.replace("**", "").replace("\u202f", " ").strip()
                    id_search  = re.search(r"\((\d+)\)", clean_name)
                    actual_id  = id_search.group(1) if id_search else p_id
                    final_name = re.sub(r"\s*\(\d+\)", "", clean_name).strip()
                    players.append({"id": actual_id, "name": final_name})

            teams.append({
                "user":    row_player_id,   # Player_ID acts as the user identifier
                "agent":   row_agent,
                "players": players
            })

        print(f"✅ Loaded {len(teams)} teams from sheet for match {match_id}")
        return teams

    except Exception as e:
        return [{"error": f"Sheet read failed: {str(e)}"}]


# ── Tool 3: Master Leaderboard Calculator ─────────────────────────────────────
@tool
def calculate_leaderboard(match_id: str) -> str:
    """
    Master tool: Reads teams from Agent_Response sheet, fetches live match
    stats, calculates fantasy points using 2026 rules, and returns a
    formatted leaderboard ranked by score.
    """
    # Step 1: Get match stats from API
    stats = fetch_match_stats.run(match_id)
    if "error" in stats:
        return f"❌ Error fetching match stats: {stats['error']}"

    # Step 2: Get user selections from Google Sheet
    teams = get_user_selections_from_sheet.run(match_id)
    if not teams or "error" in teams[0]:
        return f"❌ Error reading sheet: {teams[0].get('error', 'Unknown error')}"

    # Step 3: Calculate scores
    results = []
    for team in teams:
        score = 0
        for p in team['players']:
            p_stats = stats.get(str(p['id']), {})
            score  += calculate_fantasy_points_2026(p_stats)

        results.append({
            "user":  team['user'],
            "agent": team['agent'],
            "score": score
        })

    # Step 4: Sort by score descending
    results.sort(key=lambda x: x['score'], reverse=True)

    # Step 5: Format leaderboard
    output  = "\n" + "═" * 55 + "\n"
    output += f"🏆  FANTASY LEADERBOARD (Match ID: {match_id})  🏆\n"
    output += "═" * 55 + "\n"
    output += f"{'Rank':<5} | {'Player ID':<20} | {'Agent':<15} | {'Score':<8}\n"
    output += "─" * 55 + "\n"
    for i, res in enumerate(results, 1):
        medal = ["🥇", "🥈", "🥉"][i-1] if i <= 3 else f"{i} "
        output += f"{medal:<5} | {res['user']:<20} | {res['agent']:<15} | {res['score']:<8}\n"
    output += "═" * 55

    return output



# ── Validator Agent ────────────────────────────────────────────────────────────
validator_prompt = """You are the "Fantasy Audit Chief" — an expert cricket fantasy league analyst.

YOU HAVE EXACTLY ONE TOOL: 'calculate_leaderboard'. NO OTHER TOOLS EXIST.

WORKFLOW:
1. Use the 'calculate_leaderboard' tool to get the actual scores for the match.
2. After receiving the tool data, format your final response .

ABSOLUTE RULES — zero exceptions:
- You ONLY call 'calculate_leaderboard'. NEVER call any other tool or function.

- NEVER invent, guess, or estimate any scores or player data.
- NEVER respond before 'calculate_leaderboard' has returned its output.
- NEVER add commentary, text, or explanation outside the structured output.
- If 'calculate_leaderboard' fails, return empty rankings list immediately.
- NEVER attempt to call a function named 'reasoning_summary'.
- NEVER attempt to call a any other tool other than calculate_leaderboard'.
OUTPUT INSTRUCTIONS:

   - reasoning_summary: 3-4 lines explaining environmental logic
   - players: List of   objects (rank, player_id, agent, score)


"""
from langchain.agents import create_agent

validator_agent = create_agent(
    model=model,
    tools=[calculate_leaderboard],
    # response_format=LeaderboardOutput,
    system_prompt=validator_prompt
)

# ── Read Match_ID from Sheet at the Start ─────────────────────────────────────
all_values = agent_response_sheet.get_all_values()
data_rows  = all_values[1:]  # skip header

# Get first valid match_id from the sheet
match_id = None
for row in data_rows:
    if any(row) and row[3].strip():
        match_id = str(row[3]).strip()
        break

print(f"✅ Match ID loaded from sheet: {match_id}")

# ── Run Validator Agent ────────────────────────────────────────────────────────
user_input = f"Validate the performance for match_id {match_id}."



response = validator_agent.invoke(
    {"messages": [{"role": "user", "content": user_input}]}
)



# 1. Extract the String from the Agent Response
if "messages" in response and len(response["messages"]) > 0:
    final_message = response["messages"][-1]

    # final_message might be a string or a message object with a .content attribute
    raw_content = getattr(final_message, 'content', str(final_message))

    try:
        # 2. Parse the String into a Python Dictionary
        # We find the first '{' and last '}' just in case the AI added extra text
        start_idx = raw_content.find('{')
        end_idx = raw_content.rfind('}') + 1
        json_str = raw_content[start_idx:end_idx]

        data_dict = json.loads(json_str)


        # 3. Access the 'players' or 'rankings' list (check your prompt's key name)
        # Your previous error showed "players", but your class used "rankings"
        # Let's handle both just in case:
        player_list = data_dict.get("players") or data_dict.get("rankings") or []
        print(player_list)
        # 4. Prepare Rows for Google Sheets
        rows_to_write = []
        for p in player_list:
            rows_to_write.append([
                p.get("rank"),
                p.get("player_id"),
                p.get("agent"),
                p.get("score")
            ])

        # 5. Write to Sheet
        validator_sheet = spreadsheet.worksheet("Validator_Agent")
        # Optional: Clear old data (Columns A to D)
        validator_sheet.batch_clear(["A2:D100"])

        if rows_to_write:
            validator_sheet.insert_rows(rows_to_write, 2)
            print(f"🎉 Successfully stored {len(rows_to_write)} players in Validator_Agent sheet!")
        else:
            print("⚠️ No player data found in the JSON response.")

    except Exception as e:
        print(f"❌ Failed to parse or save JSON: {e}")
        print(f"Original content was: {raw_content}")



✅ Match ID loaded from sheet: 148963


2026-03-28 05:41:18,755 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


✅ Loaded 26 teams from sheet for match 148963


2026-03-28 05:41:23,432 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


[{'rank': 1, 'player_id': 'PLR_1004', 'agent': 'Eco-Scout', 'score': 615}, {'rank': 2, 'player_id': 'PLR_1014', 'agent': 'Eco-Scout', 'score': 587}, {'rank': 3, 'player_id': 'PLR_1026', 'agent': 'Eco-Scout', 'score': 583}, {'rank': 4, 'player_id': 'PLR_1013', 'agent': 'Overall-Scout', 'score': 541}, {'rank': 5, 'player_id': 'PLR_1012', 'agent': 'Eco-Scout', 'score': 523}, {'rank': 6, 'player_id': 'PLR_1010', 'agent': 'Eco-Scout', 'score': 498}, {'rank': 7, 'player_id': 'PLR_1017', 'agent': 'Overall-Scout', 'score': 497}, {'rank': 8, 'player_id': 'PLR_1022', 'agent': 'Eco-Scout', 'score': 496}, {'rank': 9, 'player_id': 'PLR_1006', 'agent': 'Eco-Scout', 'score': 483}, {'rank': 10, 'player_id': 'PLR_1005', 'agent': 'Overall-Scout', 'score': 471}, {'rank': 11, 'player_id': 'PLR_1009', 'agent': 'Overall-Scout', 'score': 471}, {'rank': 12, 'player_id': 'PLR_1015', 'agent': 'Form-Scout', 'score': 471}, {'rank': 13, 'player_id': 'PLR_1021', 'agent': 'Overall-Scout', 'score': 471}, {'rank': 14,